# Combined six-panel Figure 2

The rows show the experimental coarsening measurements, continuum-model results, and phase-correlation analysis. Each plotting axes is physically square; colorbars sit outside those square axes.

In [1]:
from pathlib import Path
import pickle

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
import numpy as np

ROOT = Path.cwd()
DATA_FOLDER = ROOT / 'data'
if not DATA_FOLDER.is_dir():
    raise FileNotFoundError('Run this notebook from the ReplicationPackage/FIG2 folder.')

APTOS_FONT_DIRS = [
    Path('/Applications/Microsoft PowerPoint.app/Contents/Resources/DFonts'),
    Path('/Applications/Microsoft Word.app/Contents/Resources/DFonts'),
    Path('/Applications/Microsoft Excel.app/Contents/Resources/DFonts'),
]
for _font_dir in APTOS_FONT_DIRS:
    for _font_file in ('Aptos.ttf',):
        _font_path = _font_dir / _font_file
        if _font_path.exists():
            font_manager.fontManager.addfont(_font_path)

plt.rcParams.update({
    'text.usetex': True})
plt.rcParams['axes.linewidth'] = 1
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)

SAVE_FIGURE = True
OUTPUT_STEM = ROOT / 'fig' / 'Figure2_six_panel'


In [2]:
def load_pickle(path):
    with open(path, 'rb') as stream:
        return pickle.load(stream)


def moving_average(ts, values, window=10):
    ts = np.asarray(ts)
    values = np.asarray(values)
    averaged = np.convolve(values, np.ones(window) / window, mode='valid')
    return ts[window - 1:], averaged


def load_csv_with_header(path):
    data = np.loadtxt(path, delimiter=',', skiprows=1)
    return data[None, :] if data.ndim == 1 else data


experiment = load_pickle(DATA_FOLDER / 'experiment' / 'Experiment2Dcoarsening')
continuum_1d = load_pickle(DATA_FOLDER / 'continuum' / 'Phimodel1Dcoarsening')
continuum_2d = load_pickle(DATA_FOLDER / 'continuum' / 'Phimodel2Dcoarsening')

corr_path = DATA_FOLDER / 'correlation_curves_vs_tau_D10.00_mode1.csv'
decay_path = DATA_FOLDER / 'tau_decay_vs_L_D10.00_mode1.csv'
corr_data = load_csv_with_header(corr_path)
decay_data = load_csv_with_header(decay_path)

print(f'Correlation export: {corr_path.name} | shape={corr_data.shape}')
print(f'Decay export:       {decay_path.name} | shape={decay_data.shape}')


/var/folders/m6/t_w695qd1sxg1r97q8l804w80000gp/T/ipykernel_14439/92388248.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  return pickle.load(stream)


Correlation export: correlation_curves_vs_tau_D10.00_mode1.csv | shape=(568680, 6)
Decay export:       tau_decay_vs_L_D10.00_mode1.csv | shape=(5, 3)


In [3]:
# The original 2 x 2 inch figures used 0.5 x 0.5 fractional axes,
# so every plotting area below is kept at exactly 1 x 1 inch.
FIGURE_WIDTH, FIGURE_HEIGHT = 4.5, 6.8
PANEL_SIZE = 1.0  # inches
COLORBAR_WIDTH = 0.08  # inches
column_lefts = (1.0, 2.95)
row_bottoms = (4.65, 2.65, 0.65)

fig = plt.figure(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT), dpi=200)

axes = np.array([
    [fig.add_axes([left / FIGURE_WIDTH, bottom / FIGURE_HEIGHT,
                   PANEL_SIZE / FIGURE_WIDTH, PANEL_SIZE / FIGURE_HEIGHT])
     for left in column_lefts]
    for bottom in row_bottoms
])
ax_a, ax_b, ax_c, ax_d, ax_e, ax_f = axes.flat
orange = mpl.colormaps['Oranges']
experiment_color = orange(0.4)

# (a) Experiment: inverse correlation length
time = np.asarray(experiment['time'])
fwhm = np.asarray(experiment['fwhm'])
inverse_fwhm = np.divide(fwhm[1], fwhm, out=np.full_like(fwhm, np.nan), where=fwhm > 0)
ax_a.plot(time, inverse_fwhm, lw=1, color=experiment_color)
ax_a.set(xscale='log', yscale='log', xlim=(3e-2, 6), ylim=(1e-1, 2),
         xlabel='time (s)', ylabel=r'inverse correlation length $\xi^{-1}$')

N = 111          # number of sites
a = 1.0          # hexagon-center lattice spacing, set to 1 for units of a

area_per_site = np.sqrt(3) / 2 * a**2
A = N * area_per_site

L_eff = np.sqrt(A) / a

ax_a.axhline(1/L_eff,linestyle='dotted',c='k',label='$L_{\\mathrm{eff}}^{-1}$')
ax_a.set_ylim([0.9e-1,2e0])


# (b) Experiment: shear magnitude
magnitude = np.asarray(experiment['magmean'])
magnitude_time = np.arange(len(magnitude)) / 300
ax_b.plot(magnitude_time[1:], magnitude[1:], lw=1, color=experiment_color)
ax_b.set(xscale='log', yscale='log', xlim=(3e-2, 6), ylim=(3e-2, 3.1e-1),
         xlabel='time (s)', ylabel=r'shear magnitude $\langle |S| \rangle$')

# (c, d) Continuum data: 1D circles and 2D squares
marker_stride = 8
marker_size = 10
alphas_1d = np.asarray(continuum_1d['alphas'])[:len(continuum_1d['fwhms'])]
colors_1d = orange(alphas_1d / np.max(alphas_1d))
for i, alpha in enumerate(alphas_1d):
    x, y = moving_average(2 * alpha * continuum_1d['times'][i] / 20,
                          3 / np.sqrt(alpha) / continuum_1d['fwhms'][i])
    ax_c.scatter(x[::marker_stride], y[::marker_stride], edgecolor='k',
                 facecolor=colors_1d[i], s=marker_size, lw=0.2)
    x, y = moving_average(alpha * continuum_1d['times'][i], continuum_1d['mags'][i])
    ax_d.scatter(x[::marker_stride], y[::marker_stride], edgecolor='k',
                 facecolor=colors_1d[i], s=marker_size, lw=0.2)

alphas_2d = np.asarray(continuum_2d['alphas'])
colors_2d = orange(alphas_2d / np.max(alphas_2d))
for i, alpha_raw in enumerate(alphas_2d):
    alpha = np.sqrt(alpha_raw)
    x, y = moving_average(4 * alpha * continuum_2d['times'][i],
                          0.08 / continuum_2d['fwhms'][i])
    ax_c.scatter(x[::marker_stride], y[::marker_stride], marker='s', edgecolor='k',
                 facecolor=colors_2d[i], s=marker_size, lw=0.2)
    x, y = moving_average(10 * alpha * continuum_2d['times'][i], continuum_2d['mags'][i])
    ax_d.scatter(x[::marker_stride], y[::marker_stride], marker='s', edgecolor='k',
                 facecolor=colors_2d[i], s=marker_size, lw=0.2)

ax_c.set(xscale='log', yscale='log', xlim=(3e1, 2e3), ylim=(2e-3, 2e-1),
         xlabel=r'$\sqrt{\alpha}t$', ylabel=r'inverse correlation length $\xi^{-1}$')
ax_d.set(xscale='log', yscale='log', xlim=(3e1, 2e3), ylim=(1e-3, 2e2),
         xlabel=r'$\sqrt{\alpha}t$', ylabel=r'shear magnitude $\langle |S| \rangle$')
scaling_time = np.logspace(-5, 5, 100)
ax_c.plot(scaling_time, 1.45 * scaling_time**-0.5, 'k--', lw=1)
ax_c.plot(scaling_time, 31 * scaling_time**(-5 / 4), 'k--', lw=1)
legend_handles = [
    Line2D([0], [0], marker='o', color='k', markerfacecolor='w', markersize=4,
           label=r'$1\mathrm{D}$', linestyle='none'),
    Line2D([0], [0], marker='s', color='k', markerfacecolor='w', markersize=4,
           label=r'$2\mathrm{D}$', linestyle='none'),
]
ax_c.legend(handles=legend_handles, loc='lower left', fontsize=8, frameon=False)
ax_d.legend(handles=legend_handles, loc='lower right', fontsize=8, frameon=False)
alpha_map = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, np.max(alphas_2d)), cmap=orange)
position_d = ax_d.get_position()
cax_d = fig.add_axes([position_d.x1, position_d.y0,
                      COLORBAR_WIDTH / FIGURE_WIDTH, position_d.height])
cbar_d = fig.colorbar(alpha_map, cax=cax_d)
cbar_d.ax.set_title(r'$\alpha$', pad=4)
cbar_d.set_ticks([0, 200], labels=[r'$0$', r'$10$'])
cbar_d.ax.tick_params(labelsize=8)

# (e) Correlation curves
# Columns: L, tau, C_data, legacy C_theory, D, mag_sq.
L, tau, correlation, _, D_values, mag_sq = corr_data[:, :6].T

unique_L = np.sort(np.unique(L))
print(unique_L)
n_L = len(unique_L)
corr_cmap = mpl.colors.ListedColormap(mpl.colormaps['Greys'](np.linspace(0.25, 1, n_L)))
corr_norm = mpl.colors.BoundaryNorm(np.arange(n_L + 1) - 0.5, n_L)
corr_xlim = (10, 10000)
n_visible_markers = 20
endpoint_exclusion_fraction = 0.50  # Use only the first half of each finite record.
for i, L_value in enumerate(unique_L):
    mask = np.isclose(L, L_value)
    order = np.argsort(tau[mask])
    tau_i = tau[mask][order]
    correlation_i = correlation[mask][order]
    D_i = D_values[mask][order]
    mag_sq_i = mag_sq[mask][order]
    q1 = 2 * np.pi / L_value
    theory_i = np.exp(
        -3 * D_i / (q1**2 * mag_sq_i * L_value)
        * (q1**2 * tau_i / 3 - 1 + np.exp(-q1**2 * tau_i / 3))
    )
    curve_color = corr_cmap(corr_norm(i))
    ax_e.plot(tau_i, theory_i, color=curve_color, lw=1, zorder=-20000)

    # Choose points nearest to logarithmically spaced target times.
    reliable_tau_max = min(
        corr_xlim[1], (1 - endpoint_exclusion_fraction) * tau_i[-1]
    )
    marker_tau_targets = np.geomspace(
        corr_xlim[0], reliable_tau_max, n_visible_markers
    )
    right_indices = np.searchsorted(tau_i, marker_tau_targets, side='left')
    right_indices = np.clip(right_indices, 1, len(tau_i) - 1)
    left_indices = right_indices - 1
    use_left = (
        np.abs(tau_i[left_indices] - marker_tau_targets)
        <= np.abs(tau_i[right_indices] - marker_tau_targets)
    )
    sample_indices = np.unique(np.where(use_left, left_indices, right_indices))
    ax_e.scatter(tau_i[sample_indices], correlation_i[sample_indices],
                 edgecolor='k', facecolor=curve_color,
                 s=marker_size, lw=0.2, zorder=-43)
ax_e.set(xlim=corr_xlim, ylim=(0, 1.2), xlabel=r'time lag $\tau$',
         ylabel=r'phase autocorrelation $|C(\tau)|$')
ax_e.set_xscale('log')
ax_e.set_xticks([10,100,1000,10000])
ax_e.set_yticks([0, 0.5, 1.0])
corr_map = mpl.cm.ScalarMappable(cmap=corr_cmap, norm=corr_norm)
position_e = ax_e.get_position()
cax_e = fig.add_axes([position_e.x1, position_e.y0,
                      COLORBAR_WIDTH / FIGURE_WIDTH, position_e.height])
cbar_e = fig.colorbar(corr_map, cax=cax_e, ticks=np.arange(n_L))
cbar_e.ax.set_yticklabels([str(int(round(value))) for value in unique_L])

cbar_e.ax.tick_params(labelsize=8)
cbar_e.ax.set_title(r'system size $L$', fontsize=9, pad=4)

# (f) Decay-time scaling
decay_L, tau_decay = decay_data[:, :2].T
order = np.argsort(decay_L)
decay_L, tau_decay = decay_L[order], tau_decay[order]
decay_color_indices = [int(np.argmin(np.abs(unique_L - value))) for value in decay_L]
decay_point_colors = [corr_cmap(corr_norm(index)) for index in decay_color_indices]
ax_f.scatter(decay_L, tau_decay, facecolor=decay_point_colors, edgecolor='k',
             s=marker_size*2, lw=0.2, )
if len(decay_L) >= 2 and np.all(decay_L > 0) and np.all(tau_decay > 0):
    _, log_prefactor = np.polyfit(np.log(decay_L), np.log(tau_decay), 1)
    fit_L = np.linspace(np.min(decay_L), np.max(decay_L), 300)
    ax_f.plot(fit_L, np.exp(log_prefactor*1.2) * fit_L**3, 'k--', lw=1,
              label=rf'$\propto L^3$',zorder=-10)
ax_f.set(xscale='log', yscale='log', xlabel=r'system size $L$',
         ylabel=r'decay time $\tau_{1/e}$')
ax_f.set_xticks([10,20, 40, 100], labels=[r'$10$','$20$', r'$40$', r'$100$'])
ax_f.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax_f.legend(loc='upper left', fontsize=6, frameon=False, borderaxespad=0.35)

# A physical 1:1 box aspect is applied to every panel; colorbars are separate axes.
for ax in axes.flat:
    ax.set_box_aspect(1)
    ax.tick_params(labelsize=12)

# Place letters outside each complete rendered panel, including axis labels.
PANEL_LETTER_FONT = ['Aptos', 'Arial', 'DejaVu Sans']
PANEL_LETTER_WEIGHT = 'normal'
PANEL_LETTER_X_PAD = 0.02
PANEL_LETTER_Y_PAD = 0.04

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
panel_bboxes = [
    [ax.get_tightbbox(renderer).transformed(fig.transFigure.inverted()) for ax in row]
    for row in axes
]
panel_letter_x = [min(panel_bboxes[row_index][col_index].x0 for row_index in range(axes.shape[0])) - PANEL_LETTER_X_PAD
                  for col_index in range(axes.shape[1])]
for row_index, row in enumerate(axes):
    for col_index, ax in enumerate(row):
        panel_label = 'abcdef'[row_index * axes.shape[1] + col_index]
        panel_bbox = panel_bboxes[row_index][col_index]
        fig.text(panel_letter_x[col_index], panel_bbox.y1 + PANEL_LETTER_Y_PAD, panel_label,
             fontsize=16, fontfamily=PANEL_LETTER_FONT,
             fontweight=PANEL_LETTER_WEIGHT,
             usetex=False, ha='right', va='bottom')

if SAVE_FIGURE:
    OUTPUT_STEM.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_STEM.with_suffix('.pdf'), bbox_inches='tight', transparent=True)


[ 10.  30.  50.  70. 100.]
